# ⚙️ [1/3] Check your Google Drive

        DiverScan-release
        │
        ├── data
        │        └── yourdata_folder
        │
        ├── projects
        │        └── yourproject_folder
        │                ├── dataset
        │                ├── optuna
        │                └── training
        │                        ├── epochXX_for_visualization (🟢  `epoch_visualization_folder`)
        │                        ├── best_epoch.txt
        │                        └── ...
        │
        └── utils

Ensure you have the `epochXX_for_visualization` folder.

# ⚙️ [2/3] Authorize this notebook

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [3/3] Load data and visualize the result

1.   Input followings, then select `Runtime` -> `Run all`.

2.   See your result!! Here we use the test set.

In [ ]:
#@title ⚙️ Setting

import os
import sys
from pathlib import Path
import pickle
import gzip
import random
import numpy as np
import pandas as pd

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Running in Google Colab environment.")
except ImportError:
    # Not running in Google Colab environment
    pass

# For reproductivity
seed = 0
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)

project_folder = "/content/drive/MyDrive/DiverScan-release/projects/fruitfly" #@param {type:"string"}
training_folder_name = "training" #@param {type:"string"}
epoch_folder_name = "epoch49_for_visualization" #@param {type:"string"}

project_folder = Path(project_folder)
training_folder = project_folder / training_folder_name
epoch_visualization_folder = training_folder / epoch_folder_name
os.chdir(epoch_visualization_folder)  # Check that it exists


def read_settings_file(path):
    values = {}
    with open(path, "r") as f:
        for line in f:
            key, separator, value = line.rstrip("\n").partition(": ")
            if separator:
                values[key] = value
    return values


# The training folder records which optuna folder it was trained from.
os.chdir(training_folder)
print(training_folder)
loaded_values = read_settings_file("learning_settings.txt")
optuna_folder_name = loaded_values.get("optuna_folder_name") or Path(loaded_values["optuna_folder"]).name
optuna_folder = project_folder / optuna_folder_name

os.chdir(optuna_folder)
print(optuna_folder)
loaded_values = read_settings_file("learning_settings.txt")
model_type = loaded_values["model_type"]
task_type = loaded_values["task_type"]
node_shuffle = loaded_values["node_shuffle"]
number_of_attention_branches = int(loaded_values["number_of_attention_branches"])
dataset_folder = project_folder / "dataset"

# The code to import is the snapshot 01_build_dataset copied into the project.
utils_snapshot = loaded_values.get("utils_snapshot")
if utils_snapshot:
    utils_folder = str(project_folder / utils_snapshot)
else:
    utils_folder = loaded_values["utils_folder"]
    print("No code snapshot in this project; using", utils_folder)
if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import postprocessing as postprocessing_module

# Change current directory
if os.path.isdir(dataset_folder):
    os.chdir(dataset_folder)
    print(dataset_folder)
    loaded_values = read_settings_file("settings_of_preprocessing.txt")
    if "moving_window_size" in loaded_values:
        # Show the moving window size
        print("moving window size:", loaded_values["moving_window_size"], "frames")
    with gzip.open("dataset_unstandardized_for_post_analysis.pkl", "rb") as file:
        post_analysis_dictionary = pickle.load(file)

# Change current directory and load the data
os.chdir(epoch_visualization_folder)

with gzip.open("prediction_attention_namelist.pkl", "rb") as file:
    loaded_data = pickle.load(file)
attentions = loaded_data["attentions"]
class_name_list = loaded_data["class_name_list"]
feature_name_list = loaded_data["feature_name_list"]
edge_data_name_list = loaded_data["edge_data_name_list"]
if model_type == "multi":
    pred_masks = loaded_data["pred_masks"]

with gzip.open("dictionary_for_visualization.pkl", "rb") as file:
    dictionary = pickle.load(file)

In [ ]:
#@title 1️⃣ Model performance
#@markdown You can check the performance of the attention branches.
from IPython.display import display


if task_type == "classification":

    print("⭐️ Macro averaged accuracy (test set) ⭐️")
    df = pd.read_excel("macro_averaged_accuracy.xlsx")
    df.rename(
        columns=lambda x: '' if isinstance(x, str) and x.startswith('Unnamed') else x,
        inplace=True
    )
    display(df)
    print("\n\n⭐️ Classification report (test set) ⭐️\n")

    classification_report_dictionary = pd.read_excel("classification_report.xlsx", sheet_name=None)
    postprocessing_module.show_classification_report(
        classification_report_dictionary,
        number_of_attention_branches,
    )
    print("\nFor details, see the document of sklearn.metrics.classification_report. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html")

In [ ]:
#@title 2️⃣ PCA of attention timeseries
#@markdown You can check which attention branches are distinctive and which are similar to each other.

flatten_attentions = attentions.transpose(1, 0, 2, 3).reshape(attentions.shape[1], -1)  # (number_of_attention_branches, files*N*T)

postprocessing_module.create_interactive_pca_plot(
    flatten_attentions,
    "Default",
    scale=20,
)

In [ ]:
#@title 3️⃣ Visualization of attention

default_filename_list = dictionary[class_name_list[0]].keys()
_, number_of_nodes, _ = dictionary[class_name_list[0]][list(default_filename_list)[0]]["attentions"].shape

postprocessing_module.make_plots_and_gui(
    dictionary,
    model_type,
    class_name_list,
    default_filename_list,
    number_of_attention_branches,
    feature_name_list,
    number_of_nodes,
    dpi=100,
    xlim=None,  # e.g., (0, 100) or None
    ylim=None,  # e.g., (0, 100) or None
)

In [ ]:
#@title 4️⃣ Interactive plot
#@markdown Select the left or right plot of 3️⃣ which you want to create a interactive one.

postprocessing_module.make_interactive_plot_and_gui(
    dictionary,
    model_type,
    sampling_interval=10,
)

In [ ]:
#@title 5️⃣ Correlation and Hellinger distance
#@markdown You can see potential features that contribute to attention maps.
#@markdown - Select the option to show and run this cell (hit left ▶️ of this cell).
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown - Hellinger distance is a measure of the similarity between two distributions.
#@markdown - When the Hellinger distance is 0, the two distributions are identical, and when it is 1, they are completely different.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show = "Correlation (basic features)" #@param ["Correlation (basic features)", "Correlation in highlighted segments (basic features)", "Hellinger distance in highlighted segments (basic features)", "[Multi mode] Correlation (differences of basic features between nodes)", "[Multi mode] Correlation in highlighted segments (differences of basic features between nodes)", "[Multi mode] Hellinger distance in highlighted segments (differences of basic features between nodes)", "[Multi mode] Correlation (edge features)", "[Multi mode] Correlation in highlighted segments (edge features)", "[Multi mode] Hellinger distance in highlighted segments (edge features)", "[Multi mode] Correlation (incident edge features per node)", "[Multi mode] Correlation in highlighted segments (incident edge features per node)", "[Multi mode] Hellinger distance in highlighted segments (incident edge features per node)", "[Multi mode] Correlation (MST features)", "[Multi mode] Correlation in highlighted segments (MST features)", "[Multi mode] Hellinger distance in highlighted segments (MST features)", "[Multi mode] Correlation (geometric features)", "[Multi mode] Correlation in highlighted segments (geometric features)", "[Multi mode] Hellinger distance in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result(
    show,
    class_name_list,
    number_of_attention_branches=number_of_attention_branches,
    model_type=model_type,
)


In [ ]:
#@title 6️⃣ Histogram of features in the highlighted segments
#@markdown You can see distribution differences of feature values of two conditions in the highlighted segments.
#@markdown - Select the option to show and run this cell (hit left ▶️ of this cell).
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown
#@markdown **[Multi mode] indicators** (available when model_type == "multi"):
#@markdown - `differences of basic features between nodes`: sum of absolute differences between each node's feature and those of the other nodes. Higher when the node behaves differently from the rest of the group.
#@markdown - `edge features`: mean / variance over all edges (broadcast to every node as the same value). Reflects the overall cohesion and variability of the group.
#@markdown - `incident edge features per node`: per-node sum / min / max / variance of the edges incident to the node. Distinguishes isolated individuals from central individuals on a node-by-node basis.
#@markdown - `MST features`: degree / is_leaf / incident sum / incident max on the Minimum Spanning Tree built from the distance edge. Captures structural roles such as leaf, hub, or bridge.
#@markdown - `geometric features`: distance to the group centroid / local density / head-direction alignment computed from (x, y) coordinates (only when coordinates are available).

show = "Histogram in highlighted segments (basic features)" #@param ["Histogram in highlighted segments (basic features)", "[Multi mode] Histogram in highlighted segments (differences of basic features between nodes)", "[Multi mode] Histogram in highlighted segments (edge features)", "[Multi mode] Histogram in highlighted segments (incident edge features per node)", "[Multi mode] Histogram in highlighted segments (MST features)", "[Multi mode] Histogram in highlighted segments (geometric features)"]


postprocessing_module.show_selected_result(
    show,
    class_name_list,
    feature_name_list=feature_name_list,
    dpi=100,
    model_type=model_type,
)


In [ ]:
#@title 7️⃣ [Multi mode] Correlation (based on the distance between nodes)
#@markdown 7️⃣-1. Calculate clusters.
#@markdown - Any group of nodes connected by edges whose weights fall below the threshold is considered a single cluster.
#@markdown - If you want to set thresholds, change "userset_thresholds" from "None" and run this cell (hit left ▶️ of this cell). Azimuth is not used for this calculation. Please fill it with any number.

if model_type == "multi":
    angle_keywords = ['Azimuth', 'Head direction', 'azimuth', 'head direction']
    edge_data_name_list = post_analysis_dictionary['edge_data_name_list']
    print("Edge features:", edge_data_name_list)

    angle_feature_indices = [
        i for i, name in enumerate(edge_data_name_list)
        if any(keyword in name for keyword in angle_keywords)
    ]

    thresholds = []
    for idx, i in enumerate(np.abs(post_analysis_dictionary["dictionary"]["edge_data"]).transpose(1, 0, 2, 3)):
        if idx in angle_feature_indices:
            thresholds.append(3.14/2)
        else:
            thresholds.append(float(np.mean(i)/4.0))

    userset_thresholds = None #@param {type:"raw"}

    if userset_thresholds:
        if not isinstance(userset_thresholds, list):
            raise ValueError('Check the input format of "userset_thresholds". Use the list format. eg., [15.0, 3.14, 3.14, 10.0, 10.0]')
        if len(userset_thresholds) != len(edge_data_name_list):
            raise ValueError(
                """Check the input format of "userset_thresholds". The list in "userset_thresholds"
            should include items corresponding to the number of the edge features.
            e.g., [15.0, 3.14, 3.14, 10.0, 10.0]
            for Edge features ["Distance between nodes", "Azimuth difference between nodes", "Head direction difference between nodes", "Bodypart1-to-bodypart1 distance between nodes", "Bodypart1-to-bodypart8 distance between nodes"]"""
            )
        for item in userset_thresholds:
            if not isinstance(item, float):
                raise ValueError('Check the input format of "userset_thresholds". Use only float. eg., "10.0": OK. "10": error.')

        thresholds = userset_thresholds

    print("thresholds:", thresholds)

    if os.path.isfile("correlation_cluster.xlsx"):
        os.remove("correlation_cluster.xlsx")
    if os.path.isfile("correlation_cluster_in_highlighted_segments.xlsx"):
        os.remove("correlation_cluster_in_highlighted_segments.xlsx")
    # if os.path.isfile("hellinger_distance_cluster.xlsx"):
    #     os.remove("hellinger_distance_cluster.xlsx")

    saver = postprocessing_module.ClusterSaver(
        post_analysis_dictionary=post_analysis_dictionary,
        attentions=attentions,
        pred_masks=pred_masks,
        thresholds=thresholds,
    )
    saver.save_all()

In [ ]:
#@title 7️⃣-2 Show clusters.
#@markdown You can see potential features that contribute to attention maps.
#@markdown - Select the option to show and run this cell (hit left ▶️ of this cell).
#@markdown - "Highlighted segments" are segments where the attention value exceeds the reciprocal of (the time length)(the number of nodes). 1/TN.
#@markdown - Hellinger distance is a measure of the similarity between two distributions.
#@markdown - When the Hellinger distance is 0, the two distributions are identical, and when it is 1, they are completely different.

show = "[Multi mode] Correlation in highlighted segments (cluster number based on the distance between nodes)" #@param ["[Multi mode] Correlation (cluster number based on the distance between nodes)", "[Multi mode] Correlation in highlighted segments (cluster number based on the distance between nodes)"]

#, "[Multi mode] Hellinger distance in highlighted segments (cluster number of edge features)"]

postprocessing_module.show_selected_result(
    show,
    class_name_list,
    number_of_attention_branches=number_of_attention_branches,
    model_type=model_type,
)


In [ ]:
#@title 7️⃣-3 Show configurations
#@markdown See how attention relates to the spatial "configuration" of the nodes.
#@markdown - A configuration is the connection pattern of the proximity graph (two nodes are connected when the distance between them is within the threshold), classified up to node relabeling (i.e., graph isomorphism). For example, 4 nodes yield 11 configurations and 5 nodes yield 34.
#@markdown - This is available only for multi-node models with 5 or fewer nodes; with more nodes there are too many possible configurations to interpret.
#@markdown - ⚠️ Run "7️⃣ [Multi mode] Correlation (based on the distance between nodes)" first so that the distance thresholds are computed.
#@markdown - The legend below shows what each configuration (Config. 1, 2, ...) looks like as a graph.

show = "[Multi mode] Correlation in highlighted segments (configuration patterns based on the distance between nodes)" #@param ["[Multi mode] Correlation (configuration patterns based on the distance between nodes)", "[Multi mode] Correlation in highlighted segments (configuration patterns based on the distance between nodes)"]


postprocessing_module.show_selected_result(
    show,
    class_name_list,
    number_of_attention_branches=number_of_attention_branches,
    model_type=model_type,
)


In [ ]:
#@title 8️⃣ [Multi mode, node shuffle=False] Node attention
#@markdown You can see which node gets high attention value.
from IPython.display import display


if model_type == "multi": #and node_shuffle == "False":

    df = pd.read_excel("node_attention.xlsx")
    df.rename(
        columns=lambda x: '' if isinstance(x, str) and x.startswith('Unnamed') else x,
        inplace=True
    )
    display(df)

In [ ]:
#@title 9️⃣ Model prediction of all files in the test set
#@markdown "True" indicates that the model prediction is correct, while "False" indicates it is incorrect. For more details, go to the `epochXX_for_visualization` folder.

prediction_true_false = pd.read_excel("prediction_true_false.xlsx", sheet_name=None)

postprocessing_module.show_table_and_gui(
    prediction_true_false,
    list(prediction_true_false.keys()),
    dropdown_name="condition",
)

In [ ]:
#@title Make summary tables (SummaryTable.xlsx)

import subprocess
from pathlib import Path

SCRIPT_PATH  = Path(utils_folder) / "make_summary_table.py"
INPUT_DIR    = Path(epoch_visualization_folder)
OUTPUT_XLSX  = INPUT_DIR / "SummaryTable.xlsx"

CLASS_NAMES = class_name_list
NUM_ATTENTIONS = number_of_attention_branches
MODEL_TYPE = model_type
# --------------------------------------

cmd = [
    "python",
    str(SCRIPT_PATH),
    str(INPUT_DIR),
    "--class-names", str(CLASS_NAMES),
    "--num-attentions", str(NUM_ATTENTIONS),
    "--model-type", str(MODEL_TYPE),
    "-o", str(OUTPUT_XLSX),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("Saved:", OUTPUT_XLSX.resolve())
